<a href="https://colab.research.google.com/github/alban314/TIPE/blob/main/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Installation des dépendances**

In [1]:
!pip install noise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for noise: filename=noise-1.2.2-cp312-cp312-linux_x86_64.whl size=56629 sha256=0ccddee8a98c22435e569fdd3185615b1f2aeba2a8f92312eb66cdf011b11177
  Stored in directory: /root/.cache/pip/wheels/78/71/a2/47a0c6acdeb8f7a2f4e69067d3c737219e36414136441a1ef8
Successfully built noise


In [2]:
import csv
import numpy as np
import plotly.offline as go_offline
import plotly.subplots as sub
import plotly.graph_objects as go
import random
import noise
import time
import heapq
from scipy.interpolate import griddata, RegularGridInterpolator
from matplotlib import pyplot as plt

# **Module d'affichage/génération de terrain**
A rajouter :


*   Importer un terrain ?
*   Améliorer l'affichage des cartes : légende. Permettre de n'afficher que 2D ou 3D ?




**Génération du terrain :**

In [3]:


def random_generating(n=100):
    """Valeur d'entrée : n : nombre de points générés aléatoirement
       Valeur de sortie : array numpy x,y et z générées aléatoirement de tailles n représentants n points"""
    x = np.random.randint(0, 101, size=n)
    y = np.random.randint(0, 101, size=n)
    z = np.random.randint(0, 101, size=n)
    return x, y, z


def pre_generated_terrain(liste_xyz):
    """Valeur d'entrée : liste des points (x,y,z)
       Valeur de sortie : array numpy des x,y et z séparés"""
    liste_np = np.asarray(liste_xyz)
    x = liste_np[:, 0]
    y = liste_np[:, 1]
    z = liste_np[:, 2]
    return x, y, z


def perlin_grid(n=100, scale=20.0, octaves=6):
    """
    Génère une grille de terrain via le bruit de Perlin.
    Retourne : x, y, z
    """
    x_list = np.arange(n)
    y_list = np.arange(n)

    z = np.zeros((n, n))

    for i in range(n):     # i correspond à l'index de x (lignes)
        for j in range(n): # j correspond à l'index de y (colonnes)
            z[i, j] = noise.pnoise2(i / scale, j / scale, octaves=octaves)

    return x_list, y_list, z


def generation_fonction(fonction, min_x=-10, min_y=-10, max_x=10, max_y=10, n=100):
    # Création des vecteurs d'axes
    x_range = np.linspace(min_x, max_x, n)
    y_range = np.linspace(min_y, max_y, n)

    X, Y = np.meshgrid(x_range, y_range, indexing='ij')

    Z = fonction(X, Y)

    return x_range, y_range, Z

def fonction_test_1(x, y):
    return np.cos(x)**2 + np.sin(y)**2


def populate_interpolation_points(n, x, y, z, method='linear'):
    # Grille régulière de destination
    xi = np.linspace(np.min(x), np.max(x), n)
    yi = np.linspace(np.min(y), np.max(y), n)

    Xi, Yi = np.meshgrid(xi, yi, indexing='ij')


    if np.ndim(z) == 2:
        # Si le terrain fourni est déjà une grille 2D, on doit générer ses coordonnées et tout aplatir
        X_mesh, Y_mesh = np.meshgrid(x, y, indexing='ij')
        points = (X_mesh.ravel(), Y_mesh.ravel())
        values = z.ravel()
    else:
        # Si ce sont des points aléatoires 1D (ex: de random_generating), on les utilise directement
        points = (x, y)
        values = z
    zi = griddata(points, values, (Xi, Yi), method=method)

    return xi, yi, zi

**Affichage du terrain /affichage de points ou droites**

In [4]:
# Visualisation du terrain en 2D
def terrain_2D(x, y, z, fig):
    """
    Valeur d'entrée : listes/arrays 1D x, y, matrice 2D z, figure fig
    Valeur de sortie : NULL (modifie la fig par référence)
    """
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)

    # CORRECTION : On applique .T (transposée) à z pour s'aligner sur le comportement de Plotly
    fig.add_trace(go.Contour(z=z.T, x=x, y=y, showscale=False, connectgaps=True), row=1, col=1)

    fig.update_layout(
        xaxis=dict(range=[x_min, x_max]),
        yaxis=dict(range=[y_min, y_max])
    )

# Visualisation du terrain en 3D
def terrain_3D(x, y, z, fig):
    """
    Valeur d'entrée : listes/arrays 1D x, y, matrice 2D z, figure fig
    Valeur de sortie : NULL (modifie la fig par référence)
    """
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)

    # CORRECTION : On applique .T à z ici aussi
    fig.add_trace(go.Surface(z=z.T, x=x, y=y), row=1, col=2)

    fig.update_layout(
        scene=dict(
            aspectratio=dict(x=2, y=2, z=0.5),
            xaxis=dict(range=[x_min, x_max]),
            yaxis=dict(range=[y_min, y_max])
        )
    )

# Génération de la représentation du terrain
def generation_de_la_carte(x, y, z, view2D=True, view3D=True):
    fig = sub.make_subplots(rows=1, cols=2, specs=[[{'type': 'xy'}, {'type': 'scene'}]])
    if view2D:
        terrain_2D(x, y, z, fig)
    if view3D:
        terrain_3D(x, y, z, fig)
    return fig

def export_figure(fig, nom="terrain"):
    go_offline.plot(fig, filename=nom + '.html', validate=True, auto_open=True)


def data_export(x, y, z, nom="terrain"):
    """
    Exporte l'ensemble de la grille de terrain au format CSV (X, Y, Z)
    """
    with open(nom + '.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Ajout d'une ligne d'en-tête pour la clarté
        writer.writerow(['X', 'Y', 'Z'])

        for i in range(len(x)):
            for j in range(len(y)):
                writer.writerow([x[i], y[j], z[i, j]])

def affichage_point(x, y, z, fig, view2D=True, view3D=True, color='red', offset=2, size=8, name="Point d'intérêt", flatten=True):
    """
    Ajoute des points spécifiques sur les graphiques 2D et 3D existants.
    """
    if flatten:
        x = x.flatten()
        y = y.flatten()
        z = z.flatten()

    if view2D:
        fig.add_trace(
            go.Scatter(
                x=x, y=y,
                mode='markers',
                name=name,
                marker=dict(size=size, color=color)
            ),
            row=1, col=1
        )

    if view3D:
        z_offset = np.array(z) + offset

        fig.add_trace(
            go.Scatter3d(
                x=x, y=y, z=z_offset,
                mode='markers',
                name=name,
                marker=dict(
                    size=size,
                    color=color,
                    opacity=0.9,
                    line=dict(width=2, color='white')
                )
            ),
            row=1, col=2
        )

def droite_generation(x, y, z, fig, view2D=True, view3D=True, color='red', name="Ligne"):
    if view2D:
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color=color, width=2), name=name), row=1, col=1)
    if view3D:
        fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color=color, width=2), name=name), row=1, col=2)

def affichage(x, y, z, view2D=True, view3D=True, export=True, nom_export="terrain"):
    fig = generation_de_la_carte(x, y, z, view2D, view3D)
    if export:
        export_figure(fig, nom_export)
    return fig

**Autre**

In [5]:
def generation(type="random", interpolation_method="linear", n=100, scales=20.0, octaves=6,
               liste_xyz=[], export=True, nom_export="data_record",
               fonction=lambda x, y: x**2 + y**2, intervalle=(-10, 10)):
    """
    Valeur d'entrée : type : type de génération ("random", "perlin", "pre_generated", "fonction")
    Valeur de sortie : xi, yi, zi (terrain de grille régulier) et x, y, z (données sources)
    """

    if type == "random":
        x, y, z = random_generating(n)
        xi, yi, zi = populate_interpolation_points(n, x, y, z, method=interpolation_method)
        if export:
            data_export(xi, yi, zi, nom_export)
        return xi, yi, zi, x, y, z

    if type == "perlin":
        x, y, z = perlin_grid(n, scales, octaves)
        if export:
            data_export(x, y, z, nom_export)
        return x, y, z, x, y, z

    if type == "pre_generated":
        x, y, z = pre_generated_terrain(liste_xyz)
        xi, yi, zi = populate_interpolation_points(n, x, y, z, method=interpolation_method)
        if export:
            data_export(xi, yi, zi, nom_export)
        return xi, yi, zi, x, y, z

    if type == "fonction":
        min_val, max_val = intervalle[0], intervalle[1]
        x, y, z = generation_fonction(fonction, min_x=min_val, min_y=min_val, max_x=max_val, max_y=max_val, n=n)
        if export:
            data_export(x, y, z, nom_export)
        return x, y, z, x, y, z


def generation_grille(x, y, z, n):
    """
    Sous-échantillonne une grille existante en une nouvelle grille de résolution (n x n)
    """
    x_reduced = np.linspace(np.min(x), np.max(x), n)
    y_reduced = np.linspace(np.min(y), np.max(y), n)

    x_id = np.linspace(0, len(x) - 1, n).astype(int)
    y_id = np.linspace(0, len(y) - 1, n).astype(int)

    X_mat, Y_mat = np.meshgrid(x_reduced, y_reduced, indexing='ij')

    z_reduit = z[np.ix_(x_id, y_id)]

    return X_mat, Y_mat, z_reduit

def random_color_generator():
    r = random.randint(0, 255)
    g = random.randint(0, 255)
    b = random.randint(0, 255)
    return f'rgb({r},{g},{b})'

#**Algorithme du plus court chemin**:


Rajouter :
*   Améliorer les fonctions de coût
*   Améliorer Dijkstra
*   Optimiser la plupart des fonctions pour permettre un algo plus rapide
*   Permettre d'exporter les données (temps de réalisation, cout, points de passage, point d'arrivée/départ) au format CSV



**Fonctions générales**

In [6]:
def output_export(dicti, nom="data_record"):
    with open(nom + ".csv", "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        data_name = list(dicti.keys())
        writer.writerow(data_name)
        nb_lignes = len(dicti[data_name[0]])
        for j in range(nb_lignes):
            ligne = [dicti[colonne][j] for colonne in data_name]
            writer.writerow(ligne)

def create_interpolateur(x_range, y_range, Z):
    return RegularGridInterpolator(
        (x_range, y_range),
        Z,
        method='linear',
        bounds_error=False,
        fill_value=np.nan
    )

def generation_masque(n, m):
    size = 2 * n + 1
    mask = np.zeros((size, size, 3), dtype=int)
    theta = (2 * np.pi) / m

    for i_coord in range(-n, n + 1):
        for j_coord in range(-n, n + 1):
            row = i_coord + n
            col = j_coord + n

            if i_coord == 0 and j_coord == 0:
                continue

            if np.gcd(i_coord, j_coord) == 1:
                angle = np.arctan2(i_coord, j_coord)
                if angle < 0:
                    angle += 2 * np.pi

                k = int(np.floor(angle / theta)) % m
                mask[row, col] = [i_coord, j_coord, k]

    indices_valides = np.any(mask != 0, axis=2)
    liste_deplacements = [tuple(v) for v in mask[indices_valides]]
    return liste_deplacements

def precompute_adjacency(n, masque):
    """
    Calcule pour tous les points (i,j) de la grille leurs voisins et renvoie un dictionnaire de listes des points voisins.
    """
    adjacency = {}
    for i in range(n):
        for j in range(n):
            neighbors = []
            for (di, dj, k) in masque:
                ni, nj = i + di, j + dj
                if 0 <= ni < n and 0 <= nj < n:
                    neighbors.append((ni, nj, k))
            adjacency[(i, j)] = neighbors
    return adjacency

def derivee_premiere(pt1, pt2, X_meshgrid, Y_meshgrid):
    i1, j1 = pt1[:2]
    i2, j2 = pt2[:2]
    dx = X_meshgrid[i2, j2] - X_meshgrid[i1, j1]
    dy = Y_meshgrid[i2, j2] - Y_meshgrid[i1, j1]
    return dx, dy

def distance2d(pt1, pt2, X_meshgrid, Y_meshgrid):
    i1, j1 = pt1[:2]
    i2, j2 = pt2[:2]
    dx = X_meshgrid[i2, j2] - X_meshgrid[i1, j1]
    dy = Y_meshgrid[i2, j2] - Y_meshgrid[i1, j1]
    return np.sqrt(dx**2 + dy**2)

def calculer_pente(z1, z2, d):
    return np.abs(z1 - z2) / d

def courbure(pt1, pt2, X_meshgrid, Y_meshgrid, m):
    delta_theta = abs(pt2[2]*2*np.pi/m - pt1[2]*2*np.pi/m)
    return min(delta_theta, m - delta_theta)/distance2d(pt1, pt2, X_meshgrid, Y_meshgrid)

In [13]:
def output_export(dicti, nom="data_record"):
    with open(nom + ".csv", "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        data_name = list(dicti.keys())
        writer.writerow(data_name)
        nb_lignes = len(dicti[data_name[0]])
        for j in range(nb_lignes):
            ligne = [dicti[colonne][j] for colonne in data_name]
            writer.writerow(ligne)

def create_interpolateur(x_range, y_range, Z):
    return RegularGridInterpolator(
        (x_range, y_range),
        Z,
        method='linear',
        bounds_error=False,
        fill_value=np.nan
    )

def generation_masque(n, m):
    size = 2 * n + 1
    mask = np.zeros((size, size, 3), dtype=int)
    theta = (2 * np.pi) / m

    for i_coord in range(-n, n + 1):
        for j_coord in range(-n, n + 1):
            row = i_coord + n
            col = j_coord + n

            if i_coord == 0 and j_coord == 0:
                continue

            if np.gcd(i_coord, j_coord) == 1:
                angle = np.arctan2(i_coord, j_coord)
                if angle < 0:
                    angle += 2 * np.pi

                k = int(np.floor(angle / theta)) % m
                mask[row, col] = [i_coord, j_coord, k]

    indices_valides = np.any(mask != 0, axis=2)
    liste_deplacements = [tuple(v) for v in mask[indices_valides]]
    return liste_deplacements

def precompute_adjacency(n, masque):
    """
    Calcule pour tous les points (i,j) de la grille leurs voisins et renvoie un dictionnaire de listes des points voisins.
    """
    adjacency = {}
    for i in range(n):
        for j in range(n):
            neighbors = []
            for (di, dj, k) in masque:
                ni, nj = i + di, j + dj
                if 0 <= ni < n and 0 <= nj < n:
                    neighbors.append((ni, nj, k))
            adjacency[(i, j)] = neighbors
    return adjacency

def derivee_premiere(pt1, pt2, X_meshgrid, Y_meshgrid):
    i1, j1 = pt1[:2]
    i2, j2 = pt2[:2]
    dx = X_meshgrid[i2, j2] - X_meshgrid[i1, j1]
    dy = Y_meshgrid[i2, j2] - Y_meshgrid[i1, j1]
    return dx, dy

def distance2d(pt1, pt2, X_meshgrid, Y_meshgrid):
    i1, j1 = pt1[:2]
    i2, j2 = pt2[:2]
    dx = X_meshgrid[i2, j2] - X_meshgrid[i1, j1]
    dy = Y_meshgrid[i2, j2] - Y_meshgrid[i1, j1]
    return np.sqrt(dx**2 + dy**2)

def calculer_pente(z1, z2, d):
    return np.abs(z1 - z2) / d

def courbure(pt1, pt2, X_meshgrid, Y_meshgrid, m):
    delta_theta = abs(pt2[2] - pt1[2])
    return min(delta_theta, m - delta_theta)

**Fonctions de cout**

In [14]:
def precompute_edge_costs(n, m, masque, z, X_meshgrid, Y_meshgrid, poids_cout, dist_pas, interpolateur):
    """
    Calcule les couts des différentes arrètes avant d'effectuer Dijkstra.
    Renvoie un dictionnaire dont les clés sont (i, j, ni, nj) et les valeurs sont (cout_total_no_curvature, distance_cost, pente_cost, terrain_cost).
    """
    c_p_func = poids_cout("pente")
    c_c_func_dummy = poids_cout("courbure") # Not used in cout_trajet_points, just to satisfy signature
    c_d_func = poids_cout("distance")

    # Handle optional terrain cost function
    _c_t_func_return = poids_cout("terrain")
    if callable(_c_t_func_return):
        c_t_func = _c_t_func_return
    else:
        c_t_func = lambda x: 0 # Default to 0 cost if not provided

    edge_cache = {}

    for i in range(n):
        for j in range(n):
            for (di, dj, k) in masque:
                ni, nj = i + di, j + dj
                if 0 <= ni < n and 0 <= nj < n:
                    key = (i, j, ni, nj)
                    if key not in edge_cache:
                        point1 = (i, j, 0)
                        point2 = (ni, nj, k)
                        edge_cache[key] = cout_trajet_points(
                            point1, point2, z, X_meshgrid, Y_meshgrid,
                            m, c_p_func, c_c_func_dummy, c_d_func, c_t_func, dist_pas, interpolateur
                        )

    return edge_cache

def cout_trajet_points(point1, point2, carte_donnees, X_meshgrid, Y_meshgrid, m,
                       c_p_func, c_c_func, c_d_func, c_t_func, dist_pas, interpolateur):
    i1, j1, _ = point1
    i2, j2, _ = point2

    x1, y1 = X_meshgrid[i1, j1], Y_meshgrid[i1, j1]
    x2, y2 = X_meshgrid[i2, j2], Y_meshgrid[i2, j2]

    distance_val = np.hypot(x2 - x1, y2 - y1)

    n_steps = int(round(distance_val / dist_pas))
    if n_steps == 0: n_steps = 1
    pas_reel = distance_val / n_steps

    x_coords = np.linspace(x1, x2, n_steps + 1)[1:]
    y_coords = np.linspace(y1, y2, n_steps + 1)[1:]
    points_to_interpolate = np.column_stack((x_coords, y_coords))

    z_interpolated_values = interpolateur(points_to_interpolate)

    z_values = np.empty(n_steps + 1)
    z_values[0] = carte_donnees[i1, j1]
    z_values[1:] = z_interpolated_values

    slopes = calculer_pente(z_values[1:], z_values[:-1], pas_reel)
    cost_points = cout_point((x_coords, y_coords), carte_donnees)

    # Calculate individual cost components
    distance_cost_component = c_d_func(distance_val)
    pente_cost_component = np.sum(c_p_func(slopes))
    terrain_cost_component = np.sum(c_t_func(cost_points))

    # Total cost for this segment, excluding curvature (which is added in get_cout)
    segment_total_cost_no_curvature = distance_cost_component + pente_cost_component + terrain_cost_component

    return segment_total_cost_no_curvature, distance_cost_component, pente_cost_component, terrain_cost_component

def cout_point(point, carte):
    """ Placeholder for terrain cost at a point """
    return 0

def get_cout(iv, jv, edge_cache, s, v, m, c_c_func_instance, d_accumulated_cost, X_meshgrid, Y_meshgrid):
  # edge_cache now stores (segment_total_cost_no_curvature, distance_cost_component, pente_cost_component, terrain_cost_component)
  edge_cost_no_curve_total, dist_cost_edge, pente_cost_edge, terrain_cost_edge = edge_cache[(s[0], s[1], iv, jv)]

  courbure_val = courbure(s, v, X_meshgrid, Y_meshgrid, m)
  courbure_cost_edge = c_c_func_instance(courbure_val)

  # Total cost for moving from s to v, including curvature
  current_edge_full_cost = edge_cost_no_curve_total + courbure_cost_edge

  # New total accumulated cost to reach v
  new_total_path_cost = d_accumulated_cost + current_edge_full_cost

  return new_total_path_cost, dist_cost_edge, pente_cost_edge, terrain_cost_edge, courbure_cost_edge

In [ ]:
def precompute_edge_costs(n, m, masque, z, X_meshgrid, Y_meshgrid, poids_cout, dist_pas, interpolateur):
    """
    Calcule les couts des différentes arrètes avant d'effectuer Dijkstra.
    Renvoie un dictionnaire dont les clés sont (i, j, ni, nj) (cout correspondant uniquement au parcours de l'arrête -> indépendant des angles)
    """
    c_p_func = poids_cout("pente")
    c_c_func = poids_cout("courbure")
    c_d_func = poids_cout("distance")
    c_t_func = 0 #a rajouter

    edge_cache = {}

    for i in range(n):
        for j in range(n):
            for (di, dj, k) in masque:
                ni, nj = i + di, j + dj
                if 0 <= ni < n and 0 <= nj < n:
                    key = (i, j, ni, nj)
                    if key not in edge_cache:
                        point1 = (i, j, 0)
                        point2 = (ni, nj, k)
                        edge_cache[key] = cout_trajet_points(
                            point1, point2, z, X_meshgrid, Y_meshgrid,
                            m, c_p_func, c_c_func, c_d_func, c_t_func, dist_pas, interpolateur
                        )

    return edge_cache

def cout_trajet_points(point1, point2, carte_donnees, X_meshgrid, Y_meshgrid, m,
                       c_p_func, c_c_func, c_d_func,c_t_func, dist_pas, interpolateur):
    i1, j1, _ = point1
    i2, j2, _ = point2

    x1, y1 = X_meshgrid[i1, j1], Y_meshgrid[i1, j1]
    x2, y2 = X_meshgrid[i2, j2], Y_meshgrid[i2, j2]

    distance = np.hypot(x2 - x1, y2 - y1)

    n = int(round(distance / dist_pas))
    if n == 0: n = 1
    pas_reel = distance / n

    x_coords = np.linspace(x1, x2, n + 1)[1:]
    y_coords = np.linspace(y1, y2, n + 1)[1:]
    points_to_interpolate = np.column_stack((x_coords, y_coords))

    z_interpolated_values = interpolateur(points_to_interpolate)

    z_values = np.empty(n + 1)
    z_values[0] = carte_donnees[i1, j1]
    z_values[1:] = z_interpolated_values

    slopes = calculer_pente(z_values[1:], z_values[:-1], pas_reel)
    cost_points = cout_point((x_coords, y_coords),carte_donnees)
    somme_cost_terrain = np.sum(c_t_func(cost_points))
    somme_pente_cost = np.sum(c_p_func(slopes))

    cout_total = c_d_func(distance)  + somme_pente_cost + somme_cost_terrain

    return cout_total

def cout_point(point, carte):
    """ """
    return 0

def get_cout(iv, jv, edge_cache, s,v,m, cc_func, d, X_meshgrid, Y_meshgrid):
  nd = edge_cache[(s[0], s[1], iv, jv)]
  courbure_val = courbure(s, v,X_meshgrid, Y_meshgrid, m)
  c_c_func = poids_cout("courbure")
  nd += c_c_func(courbure_val)
  new_cost = d + nd
  return new_cost

**Dijkstra**

In [15]:
def dijkstra(n, m, z, X_meshgrid, Y_meshgrid, dep, arr, masque, poids_cout, dist_pas, interpolateur):
    # Precompute all edge costs once
    edge_cache = precompute_edge_costs(n, m, masque, z, X_meshgrid, Y_meshgrid, poids_cout, dist_pas, interpolateur)

    # Precompute adjacency list once
    adjacency = precompute_adjacency(n, masque)

    pq = []
    # cout_total now stores accumulated costs for (i, j, theta)
    cout_total = np.full((n, n, m), np.inf)
    # prev will store (previous_node, {'distance': d_cost, 'pente': p_cost, 'terrain': t_cost, 'courbure': c_cost})
    prev = {}
    visited = set()

    for t in range(m):
        dep_prime = (dep[0], dep[1], t)
        cout_total[dep[0]][dep[1]][t] = 0
        heapq.heappush(pq, (0, dep_prime))

    # Instantiate the curvature cost function once for efficiency
    c_c_func_instance = poids_cout("courbure")

    while pq:
        d_accumulated_cost, s = heapq.heappop(pq) # d_accumulated_cost is the total cost to reach node s

        if s in visited:
            continue
        visited.add(s)

        if (s[0], s[1]) == (arr[0], arr[1]):
            path = []
            path_local_costs = [] # List to store dictionaries of local costs for each segment

            curr = s
            # Reconstruct path and collect local costs in reverse order
            while curr in prev:
                prev_node, edge_cost_details = prev[curr]
                path.append(curr)
                path_local_costs.append(edge_cost_details)
                curr = prev_node
            path.append(curr) # Add the starting node

            # Return path and local costs in correct order (start to end)
            return path[::-1], d_accumulated_cost, path_local_costs[::-1]

        for v in adjacency[(s[0], s[1])]:
            iv, jv, theta_v = v

            # Get new total accumulated cost and individual edge costs for moving from s to v
            new_total_path_cost, dist_cost_edge, pente_cost_edge, terrain_cost_edge, courbure_cost_edge = \
                get_cout(iv, jv, edge_cache, s, v, m, c_c_func_instance, d_accumulated_cost, X_meshgrid, Y_meshgrid)

            if new_total_path_cost < cout_total[iv][jv][theta_v]:
                cout_total[iv][jv][theta_v] = new_total_path_cost
                # Store the previous node and the costs for the edge s -> v
                prev[v] = (s, {'distance': dist_cost_edge, 'pente': pente_cost_edge,
                               'terrain': terrain_cost_edge, 'courbure': courbure_cost_edge})
                heapq.heappush(pq, (new_total_path_cost, v))

    return [], np.inf, [] # Return empty list for path_local_costs if no path found

In [ ]:
def dijkstra(n, m, z, X_meshgrid, Y_meshgrid, dep, arr, masque, poids_cout, dist_pas, interpolateur):
    # Precompute all edge costs once
    edge_cache = precompute_edge_costs(n, m, masque, z, X_meshgrid, Y_meshgrid, poids_cout, dist_pas, interpolateur)

    # Precompute adjacency list once
    adjacency = precompute_adjacency(n, masque)

    pq = []
    cout_total = np.full((n, n, m), np.inf)
    prev = {}
    visited = set()

    for t in range(m):
        dep_prime = (dep[0], dep[1], t)
        cout_total[dep[0]][dep[1]][t] = 0
        heapq.heappush(pq, (0, dep_prime))

    while pq:
        d, s = heapq.heappop(pq)

        if s in visited:
            continue
        visited.add(s)

        if (s[0], s[1]) == (arr[0], arr[1]):
            path = []
            curr = s
            while curr in prev:
                path.append(curr)
                curr = prev[curr]
            path.append(curr)
            return path[::-1], d

        for v in adjacency[(s[0], s[1])]:
            iv, jv, theta_v = v
            if v in visited:
                continue

            c_c_func = poids_cout("courbure")

            new_cost = get_cout(iv, jv, edge_cache, s,v,m, cc_func, d, X_meshgrid, Y_meshgrid)

            if new_cost < cout_total[iv][jv][theta_v]:
                cout_total[iv][jv][theta_v] = new_cost
                prev[v] = s
                heapq.heappush(pq, (new_cost, v))

    return [], np.inf

**A***

In [ ]:
def a_star(n, m, z, X_meshgrid, Y_meshgrid, dep, arr, masque, poids_cout, dist_pas, interpolateur):
  # Precompute all edge costs once
  edge_cache = precompute_edge_costs(n, m, masque, z, X_meshgrid, Y_meshgrid, poids_cout, dist_pas, interpolateur)

  # Precompute adjacency list once
  adjacency = precompute_adjacency(n, masque)

  visited = set()

  return 0;

# **Exploitation des résultats**

In [9]:
def data_dist_alti(xx, yy, zz):
    dist = [0]
    tot = list(zip(xx, yy))
    x_prec, y_prec = tot[0]
    for x, y in tot[1:]:
        d = np.sqrt((x - x_prec)**2 + (y - y_prec)**2)
        dist.append(dist[-1] + d)
        x_prec, y_prec = x, y
    return dist, zz

def graphe(x, y, name="plot"):
    plt.figure()
    plt.plot(x, y, 'o-')

    plt.xlabel("Distance")
    plt.ylabel("Altitude")
    plt.grid(True)

    plt.savefig(name + ".png")
    plt.close()

# **Tests**

In [19]:

# 10 * (x**2 + 3*y**2)*np.exp(- x**2 - y**2)

x_,y_,z_,a,b,c= generation("fonction", n =500, fonction = (lambda x, y :2*(x**2 + 3*y**2)*np.exp(- x**2 - y**2)), intervalle = (-2, 2))

fig = affichage(x_,y_,z_, export = False)

interp = create_interpolateur(x_, y_, z_)

masque = generation_masque(5,20)

for k in range(20,21):
  (x,y,z) = generation_grille(x_, y_, z_, 10*k)

  color = random_color_generator()
  for w in range(3,4):
    def poids_cout(t):
      if t == "pente":
        return lambda x: np.inf if x > 3 else x*100
      if t == "courbure":
        return lambda x: x
      if t == "distance":
        return lambda x: 0
      if t == "terrain":
        return lambda x: 0

  start_time = time.time()
  # Modified call to dijkstra to receive local_costs
  trajet, cout, local_costs = dijkstra(10*k, 20, z, x, y, (5*k,int(k*5/2),0), (5*k,int(3*k*5/2),0), masque, poids_cout, 4, interp)
  end_time = time.time()

  print(f"Temps d'exécution : {end_time - start_time:.2f} secondes")
  print(f"Coût final : {cout}")
  print("Local costs per segment:")
  for i, costs in enumerate(local_costs):
      print(f"  Segment {i+1}: {costs}") # Display local costs

  if local_costs:
      # Prepare data for CSV export
      exported_costs_data = {
          'distance': [seg['distance'] for seg in local_costs],
          'pente': [seg['pente'] for seg in local_costs],
          'terrain': [seg['terrain'] for seg in local_costs],
          'courbure': [seg['courbure'] for seg in local_costs]
      }
      output_export(exported_costs_data, nom="local_path_costs")
      print("Local costs exported to local_path_costs.csv")

  if trajet:
    xx,yy,zz_vals = [],[],[]
    for t_node in trajet:
      i,j,theta = t_node
      xx.append(x[i][j])
      yy.append(y[i][j])
      zz_vals.append(z[i][j])
    affichage_point(xx, yy, zz_vals, fig, flatten = False, offset = 0, color=color, name = "point :" +str(k))
    droite_generation(xx, yy, zz_vals, fig, color=color, name = "droite :"+str(k))
    dist_plot, alti_plot = data_dist_alti(xx,yy,zz_vals)
    graphe(dist_plot, alti_plot, name = "plot"+str(k))
  else:
    print("Aucun trajet trouvé avec ces contraintes de pente.")

export_figure(fig)


Temps d'exécution : 694.04 secondes
Coût final : 1407.1739965103716
Local costs per segment:
  Segment 1: {'distance': 0, 'pente': np.float64(1.5055985151198092), 'terrain': np.int64(0), 'courbure': np.int64(0)}
  Segment 2: {'distance': 0, 'pente': np.float64(39.86249493078467), 'terrain': np.int64(0), 'courbure': np.int64(3)}
  Segment 3: {'distance': 0, 'pente': np.float64(1.5886162905156807), 'terrain': np.int64(0), 'courbure': np.int64(3)}
  Segment 4: {'distance': 0, 'pente': np.float64(0.13372188012014208), 'terrain': np.int64(0), 'courbure': np.int64(0)}
  Segment 5: {'distance': 0, 'pente': np.float64(6.351133838933948), 'terrain': np.int64(0), 'courbure': np.int64(2)}
  Segment 6: {'distance': 0, 'pente': np.float64(5.37287840654301), 'terrain': np.int64(0), 'courbure': np.int64(10)}
  Segment 7: {'distance': 0, 'pente': np.float64(2.3033758468557126), 'terrain': np.int64(0), 'courbure': np.int64(1)}
  Segment 8: {'distance': 0, 'pente': np.float64(2.0392199245029126), 'terra

In [ ]:

# 10 * (x**2 + 3*y**2)*np.exp(- x**2 - y**2)

x_,y_,z_,a,b,c= generation("fonction", n =500, fonction = (lambda x, y :2*(np.sin(x/2)*np.sin(y)+np.cos(x/2)*np.sin(2*y))), intervalle = (-5, 5))

fig = affichage(x_,y_,z_, export = False)

interp = create_interpolateur(x_, y_, z_)

masque = generation_masque(5,10)

for k in range(10,11):
  (x,y,z) = generation_grille(x_, y_, z_, 10*k)

  color = random_color_generator()
  for w in range(3,4):
    def poids_cout(t):
      if t == "pente":
        return lambda x: np.inf if x > 3 else 100 * x
      if t == "courbure":
        return lambda x: x
      if t == "distance":
        return lambda x: 0
      if t == "terrain":
        return lambda x: 0

  start_time = time.time()
  trajet, cout = dijkstra(10*k, 10, z, x, y, (5*k,int(k*5/2),0), (5*k,int(3*k*5/2),0), masque, poids_cout, 4, interp)
  end_time = time.time()

  print(f"Temps d'exécution : {end_time - start_time:.2f} secondes")
  print(f"Coût final : {cout}")

  if trajet:
    xx,yy,zz_vals = [],[],[]
    for t_node in trajet:
      i,j,theta = t_node
      xx.append(x[i][j])
      yy.append(y[i][j])
      zz_vals.append(z[i][j])
    affichage_point(xx, yy, zz_vals, fig, flatten = False, offset = 0, color=color, name = "point :" +str(k))
    droite_generation(xx, yy, zz_vals, fig, color=color, name = "droite :"+str(k))
    dist_plot, alti_plot = data_dist_alti(xx,yy,zz_vals)
    graphe(dist_plot, alti_plot, name = "plot"+str(k))
  else:
    print("Aucun trajet trouvé avec ces contraintes de pente.")

export_figure(fig)

Temps d'exécution : 139.90 secondes
Coût final : 619.8588960178721


# **Autres idées**



*   Trouver plus de fond mathématique/informatique (creuser plus les courbes paramétriques ? Réfléchir à des nouvelles structures de données moins lourdes/plus rapides ? Simplifier encore l'utilisation de l'algo (créer une interface/appli ?))
*   Comme l'article, approcher le problème de manière statistique ?
*   Comparer plusieurs algorithmes de parcours (Dijkstra et A*) ?
*   Etudier les routes générées
*   Faire un peu de matplotib
*   Chercher d'autres articles de référence pour avoir plus de fond
*   Peut on montrer à quel point le chemin le moins couteux pour une précision m est le moins couteux pour une précision inferieur n ?

Idée : créer un interface (plus simple à utiliser)
* Option de génération
* Option de choisir sur une carte en cliquant les points de départ/arrivée
* Curseurs pour les paramètres

Idée plus mathématique:
* Montrer en quelle mesure l'algorithme evite réelement les pentes/courbures (utiliser démo Dijkstra + autre ?)

Conseils de GPT:

Tu combines :

théorie des graphes,
optimisation,
génération procédurale.

Plan possible :

Représentation du terrain.
Fonction de coût.
Algorithme de plus court chemin.
Lissage de trajectoire.
Génération automatique de réseau.

6. Concepts mathématiques à citer dans le TIPE

Voici les mots-clés “prestige” qui plaisent beaucoup en TIPE :

graphes pondérés,
Dijkstra,
A*,
Voronoï,
triangulation de Delaunay,
optimisation sous contraintes,
spline cubique,
courbes de Bézier,
champs de potentiel,
génération procédurale,
automates,
théorie des réseaux,
métriques anisotropes.


**1. Articles sur la courbure et les trajectoires réalistes**

**A. Courbes de Dubins / Reeds-Shepp**

Dubins path

Article fondateur :

L. E. Dubins,
On Curves of Minimal Length with a Constraint on Average Curvature (1957)

Très pertinent pour ton TIPE.

Pourquoi :

impose un rayon minimal de virage,
très lié aux routes réelles,
relie géométrie + optimisation.

comparer :

Dijkstra classique,
coût de courbure,
trajectoire de Dubins.

**B. Splines et lissage géométrique**

Spline interpolation

Tu peux enrichir énormément avec :

spline cubiques,
Bézier,
clothoïdes.

Les clothoïdes sont particulièrement importantes :
elles sont utilisées dans les vraies routes et voies ferrées.

**C. Clothoïdes (Euler spirals)**

Euler spiral
La courbure varie linéairement avec l’abscisse curviligne.
C’est exactement ce qu’utilisent :

autoroutes,
voies SNCF,
circuits.

Tu peux :

comparer tes routes actuelles,
montrer pourquoi les transitions sont brutales,
introduire un post-traitement géométrique réaliste.

**2. Articles sur les réseaux routiers réalistes**

Maintenant que tu sais générer UNE route, le niveau supérieur est :

générer un réseau cohérent.

**D. Théorie des graphes spatiaux**

Spatial network

Tu peux étudier :

degré moyen,
hiérarchie,
centralité,
structure organique vs grille.
E. Voronoï et urbanisme
Voronoi diagram

Très utilisé pour :

découpage urbain,
routes secondaires,
parcelles.

Tu peux :

générer des centres urbains,
connecter via Voronoï,
comparer avec ton algo.

**3. Direction très intéressante : champ de potentiel**


Tu utilises déjà des masques de coût.

Mathématiquement, tu peux reformuler ça comme :

C(γ)=∫
γ
	​

Φ(x,y,θ)ds

où Φ est un champ de potentiel.

Ça ouvre :

analogies physiques,
mécanique,
optique géométrique,
eikonal equation.
F. Fast Marching Method
Fast Marching Method

Très lié à la génération de routes.

Articles :

Sethian,
propagation de fronts,
équation d’Eikonal.

Tu peux comparer :

Dijkstra discret,
Fast marching continu.

C’est très “maths modernes”.

**4. Ce que tu peux apporter d’original**


Vu ton avancement, le vrai plus maintenant est probablement :

**A. Analyse morphologique**

Ne pas juste générer :
mais mesurer.

Par exemple :

longueur moyenne,
courbure moyenne,
énergie totale,
distribution des angles,
coût global,
fractalité éventuelle.

Ça transforme ton projet en étude scientifique.

**B. Étude paramétrique**

Faire varier :

poids de pente,
poids de courbure,
bruit du terrain.

Puis observer :

routes rectilignes,
serpentines,
contournements.

Très bon TIPE expérimental.

**C. Multi-objectifs**

Très beau sujet :

C=αC
distance
	​

+βC
pente
	​

+γC
courbure
	​


et étudier :

fronts de Pareto,
compromis.

Là tu montes énormément en niveau mathématique.

**5. Articles vraiment adaptés à TON niveau actuel**

Vu ce que tu as déjà codé, je te recommande surtout :

1. Dubins

Très compatible avec ton coût de courbure.

2. Clothoïdes

Pour le réalisme routier.

Cherche :

“Highway geometric design clothoid”
“Euler spiral road design”

4. Spatial networks

Article :
Barthelemy — Spatial Networks

Excellent pour enrichir la partie réseau.

6. Honnêtement, ce qui impressionnerait le plus en TIPE

À mon avis :

Combinaison idéale
Terrain procédural.
Coût anisotrope.
Plus court chemin.
Courbure bornée.
Lissage clothoïde.
Étude paramétrique.

Ça fait :

maths,
info,
modélisation,
visualisation,
optimisation.


# Nouvelle section